# Tutorial 04 - Exercise 02: Student Grades Analysis with NumPy

Analyses a synthetic dataset of 100 students across 5 academic modules
(`THL`, `DSS`, `DB`, `SE`, `RE`).


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/05_NumPy_Scientific_Computing/` project folder (anchored on its own notebook path) by walking upward from the
current working directory — so the notebook runs identically from the repository
root, the project folder or its `notebooks/` sub-folder. On Google Colab the
repository is cloned into `/content/APSD_TAI_LAB` automatically (opening the
notebook directly from GitHub also works). The resolved project folder is added
to `sys.path` so its local modules (e.g. `utils`, `students`) import cleanly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "05_NumPy_Scientific_Computing"
ANCHOR_REL = "notebooks/02_student_grades_analysis.ipynb"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")


### 1. Generate the Dataset


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(101)
grades = np.random.randint(0, 21, size=(100, 5))
modules = ["THL", "DSS", "DB", "SE", "RE"]
print(f"Dataset shape: {grades.shape}")


### 2. Summary Statistics per Module


In [ ]:
for i, m in enumerate(modules):
    col = grades[:, i]
    print(f"Module {m:4s}: Mean={col.mean():.2f} | Median={np.median(col):.1f} | Std={col.std():.2f} | Var={col.var():.2f}")

print(f"\nOverall Min: {grades.min()} | Overall Max: {grades.max()}")


### 3. THL Module Percentiles


In [ ]:
thl = grades[:, 0]
p25, p50, p75 = np.percentile(thl, [25, 50, 75])
print(f"THL Percentiles -> 25th: {p25:.1f}, Median: {p50:.1f}, 75th: {p75:.1f}")


### 4. Boolean Masking: Scores Between 10 and 15


In [ ]:
mask = (grades >= 10) & (grades <= 15)
filtered = grades[mask]
print(f"Grades in range [10, 15]: {filtered.size} occurrences")


### 5. Distributions and Comparison Visualizations
The boxplot uses the `tick_labels` parameter (Matplotlib >= 3.9) with a
fallback to the legacy `labels` keyword on older versions.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Plot A: Average Grades
axs[0].bar(modules, grades.mean(axis=0), color="skyblue", edgecolor="black")
axs[0].axhline(10, color="red", linestyle="--", label="Passing Mark")
axs[0].set_title("Module Average Grades", fontweight="bold")
axs[0].set_ylim(0, 20)
axs[0].legend()

# Plot B: Grade Spread Boxplot (Matplotlib 3.9+ renamed labels -> tick_labels)
box_kwargs = dict(patch_artist=True, boxprops=dict(facecolor="lightgreen"))
try:
    axs[1].boxplot(grades, tick_labels=modules, **box_kwargs)
except TypeError:
    axs[1].boxplot(grades, labels=modules, **box_kwargs)
axs[1].set_title("Module Grade Distributions", fontweight="bold")

plt.tight_layout()
plt.show()
